<a href="https://colab.research.google.com/github/ShadhaShareef/reels_categorizer/blob/main/reels_categorizer_colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instagram Saved Reels Categorizer (Colab, video mode)

Runs on Colab's free compute - much more headroom than a free web-hosting tier, so
real video download + transcription works here, not just captions.

**How to use:**
1. Run the cells below in order (Runtime -> Run all, or run each cell one at a time).
2. The last cell prints a public `https://xxxxx.gradio.live` link - open that.
3. Upload your Instagram export JSON, paste your free Gemini API key, and click Submit.
4. Download the resulting zip of category Word docs.

**Notes:**
- The gradio.live link only works while this notebook keeps running. Closing the tab
  or letting Colab disconnect (free tier: ~90 min idle timeout, ~12hr session cap)
  ends it - just re-run the notebook next time you want to use it.
- For faster runs, go to `Runtime > Change runtime type` and select a GPU (still free,
  just limited weekly quota) before running the cells - Whisper will use it automatically.
- Get a free Gemini key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
- Export your saved reels: Instagram app -> Settings -> Accounts Center -> Your information
  and permissions -> Download your information -> Saved -> JSON format -> All Time (or One year).
- Similar/near-duplicate categories (e.g. "AI Development" and "AI Productivity Tools")
  are automatically merged into a single doc before the final zip is built.


## 1. Install dependencies

In [ ]:
!pip install -q yt-dlp faster-whisper python-docx gradio requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 61.9 MB/s eta 0:00:00


## 2. Define the app

In [4]:
import io
import json
import os
import re
import tempfile
import time
import zipfile
from collections import defaultdict

import gradio as gr
import requests
import yt_dlp
from docx import Document
from faster_whisper import WhisperModel

MODEL = "gemini-3.1-flash-lite"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

URL_RE = re.compile(r"https?://(?:www\.)?instagram\.com/\S+")
REEL_PATH_RE = re.compile(r"instagram\.com/(?:reel|reels|p)/[\w-]+")

PROMPT_TEMPLATE = """You analyze Instagram reels using their caption and, when available, a \
transcript of the spoken audio. Respond with ONLY a JSON object (no markdown fences, no extra \
text before or after) with these fields:

{{
  "category": "short category name, e.g. Fitness, Recipes, Finance Tips, Comedy, Travel, Productivity",
  "summary": "1-2 sentence summary of what this reel is about",
  "main_points": ["short bullet point", "short bullet point"]
}}

Pick the category based on the actual content, don't force it into a generic bucket.
main_points should be 2-5 short, concrete takeaways. Prefer the transcript for detail when
present; use the caption to fill gaps or as the main source if there's no transcript.

Caption: {caption}

Transcript: {transcript}
"""

_whisper_model = None


def get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        try:
            import ctranslate2
            has_gpu = ctranslate2.get_cuda_device_count() > 0
        except Exception:
            has_gpu = False
        device = "cuda" if has_gpu else "cpu"
        compute_type = "float16" if has_gpu else "int8"
        print(f"Loading Whisper model on {device}...")
        _whisper_model = WhisperModel("base", device=device, compute_type=compute_type)
    return _whisper_model


def is_reel_url(url: str) -> bool:
    return bool(url and REEL_PATH_RE.search(url))


def extract_entries_from_meta_json(data: dict) -> list[dict]:
    entries, seen_urls = [], set()

    def walk(node):
        if isinstance(node, dict):
            label_values = node.get("label_values")
            if isinstance(label_values, list):
                url, caption = None, ""
                for lv in label_values:
                    if not isinstance(lv, dict):
                        continue
                    label = lv.get("label")
                    if label == "URL" and is_reel_url(lv.get("value")):
                        url = lv["value"]
                    elif label == "Caption":
                        caption = lv.get("value") or ""
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    entries.append({"url": url, "caption": caption})
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for v in node:
                walk(v)

    walk(data)
    if entries:
        return entries

    urls = set()

    def walk_loose(node):
        if isinstance(node, dict):
            for v in node.values():
                walk_loose(v)
        elif isinstance(node, list):
            for v in node:
                walk_loose(v)
        elif isinstance(node, str):
            for match in URL_RE.findall(node):
                match = match.rstrip('",)')
                if is_reel_url(match):
                    urls.add(match)

    walk_loose(data)
    return [{"url": u, "caption": ""} for u in sorted(urls)]


def download_and_transcribe(url: str) -> str:
    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            out_path = os.path.join(tmpdir, "video.mp4")
            ydl_opts = {
                "format": "mp4", "outtmpl": out_path,
                "quiet": True, "no_warnings": True, "socket_timeout": 20,
            }
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            model = get_whisper_model()
            segments, _ = model.transcribe(out_path, beam_size=1)
            return " ".join(seg.text for seg in segments).strip()
    except Exception as e:
        print(f"  Video failed for {url}: {e}")
        return ""


def extract_json(text: str) -> dict:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output")
    return json.loads(match.group(0))


def call_gemini(api_key: str, caption: str, transcript: str = "") -> dict:
    prompt = PROMPT_TEMPLATE.format(
        caption=caption[:800] or "(no caption)",
        transcript=transcript[:4000] or "(no transcript available)",
    )
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"responseMimeType": "application/json"},
    }
    resp = requests.post(f"{API_URL}?key={api_key}", json=payload, timeout=60)
    if resp.status_code == 429:
        time.sleep(20)
        resp = requests.post(f"{API_URL}?key={api_key}", json=payload, timeout=60)
    resp.raise_for_status()
    raw_text = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
    return extract_json(raw_text)


def synthesize_category_overview(api_key: str, category: str, entries: list[dict]) -> str:
    all_points = []
    for e in entries:
        all_points.extend(e.get("main_points", []))
    if not all_points:
        return "No detailed points were extracted for this category."
    points_text = "\n".join(f"- {p}" for p in all_points)
    prompt = (
        f"Here are main points collected from {len(entries)} Instagram reels, "
        f"all in the category '{category}':\n\n{points_text}\n\n"
        "Write a short synthesized overview (3-6 sentences) of the recurring "
        "themes and most important takeaways across these reels. Group related "
        "points together rather than just restating each one. Plain prose, no headers."
    )
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    resp = requests.post(f"{API_URL}?key={api_key}", json=payload, timeout=60)
    if resp.status_code == 429:
        time.sleep(20)
        resp = requests.post(f"{API_URL}?key={api_key}", json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()


def consolidate_categories(api_key: str, raw_categories: list[str]) -> dict[str, str]:
    """Ask Gemini to merge near-duplicate/overly-narrow category names (e.g.
    'AI Development', 'AI Productivity Tools', 'AI Productivity') into a
    smaller set of sensible canonical categories."""
    if len(raw_categories) <= 1:
        return {c: c for c in raw_categories}

    categories_text = "\n".join(f"- {c}" for c in raw_categories)
    prompt = (
        "Here is a list of category names that were generated separately for "
        "different pieces of content, so there's a lot of near-duplication and "
        "overly narrow variants of what's really the same broader topic (e.g. "
        "'AI Development', 'AI Education', 'AI Productivity Tools', and "
        "'AI Productivity' should probably all become one category, like 'AI & Tech').\n\n"
        f"{categories_text}\n\n"
        "Group these into a smaller set of broader, sensible categories - merge "
        "obvious near-duplicates and overly specific variants, but don't over-merge "
        "genuinely distinct topics into one bucket (e.g. 'Cooking' and 'Career Advice' "
        "should stay separate). Respond with ONLY a JSON object mapping EVERY original "
        "category name (exactly as given, as the key) to its new canonical category name "
        "(as the value). Every input category must appear as a key exactly once."
    )
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"responseMimeType": "application/json"},
    }
    resp = requests.post(f"{API_URL}?key={api_key}", json=payload, timeout=60)
    if resp.status_code == 429:
        time.sleep(20)
        resp = requests.post(f"{API_URL}?key={api_key}", json=payload, timeout=60)
    resp.raise_for_status()
    raw_text = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
    mapping = json.loads(raw_text)
    for c in raw_categories:
        mapping.setdefault(c, c)
    return mapping


def safe_filename(name: str) -> str:
    return re.sub(r"[^\w\-. ]", "_", name).strip() or "Uncategorized"


def build_doc(category: str, overview: str, entries: list[dict]) -> bytes:
    doc = Document()
    doc.add_heading(category, level=0)
    doc.add_heading("Overview", level=1)
    doc.add_paragraph(overview)
    doc.add_heading(f"Reel-by-reel breakdown ({len(entries)} reels)", level=1)
    for e in entries:
        doc.add_heading(e.get("url", "Reel"), level=2)
        p = doc.add_paragraph()
        run = p.add_run(e.get("summary", ""))
        run.italic = True
        for point in e.get("main_points", []):
            doc.add_paragraph(point, style="List Bullet")
    buf = io.BytesIO()
    doc.save(buf)
    return buf.getvalue()


def process_reels(file_obj, api_key, limit, use_video, progress=gr.Progress()):
    if not file_obj or not api_key:
        return None, "Please provide both a JSON file and a Gemini API key."

    with open(file_obj.name, "r", encoding="utf-8") as f:
        data = json.load(f)
    entries = extract_entries_from_meta_json(data)
    if limit and int(limit) > 0:
        entries = entries[: int(limit)]

    if not entries:
        return None, "Couldn't find any reel URLs in this file."

    results = []
    video_count = 0
    for i, entry in enumerate(entries):
        progress((i + 1) / len(entries), desc=f"Reel {i+1}/{len(entries)}")
        transcript = ""
        if use_video:
            transcript = download_and_transcribe(entry["url"])
            if transcript.strip():
                video_count += 1

        try:
            if transcript.strip() or entry["caption"].strip():
                r = call_gemini(api_key, entry["caption"], transcript)
                entry["category"] = r.get("category", "Uncategorized")
                entry["summary"] = r.get("summary", "")
                entry["main_points"] = r.get("main_points", [])
            else:
                entry["category"] = "Uncategorized"
                entry["summary"] = "No caption or transcript to work from."
                entry["main_points"] = []
        except Exception as e:
            entry["category"] = "Uncategorized"
            entry["summary"] = f"(Failed to process: {e})"
            entry["main_points"] = []
        results.append(entry)
        time.sleep(1)

    by_category = defaultdict(list)
    for e in results:
        by_category[e["category"]].append(e)

    raw_categories = sorted(by_category.keys())
    progress(0.95, desc="Consolidating similar categories...")
    category_map = consolidate_categories(api_key, raw_categories)

    merged_by_category = defaultdict(list)
    for raw_category, cat_entries in by_category.items():
        canonical = category_map.get(raw_category, raw_category)
        merged_by_category[canonical].extend(cat_entries)
    by_category = merged_by_category

    zip_path = os.path.join(tempfile.gettempdir(), "reel_categories.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for category, cat_entries in by_category.items():
            overview = synthesize_category_overview(api_key, category, cat_entries)
            doc_bytes = build_doc(category, overview, cat_entries)
            zf.writestr(f"{safe_filename(category)}.docx", doc_bytes)
            time.sleep(1)

    summary = (
        f"Processed {len(results)} reels into {len(by_category)} categories "
        f"(merged from {len(raw_categories)} raw labels).\n"
        f"{video_count}/{len(results)} used real video transcription"
        if use_video else
        f"Processed {len(results)} reels into {len(by_category)} categories "
        f"(merged from {len(raw_categories)} raw labels)."
    )
    return zip_path, summary

## 3. Launch (click the gradio.live link that prints below)

In [5]:
demo = gr.Interface(
    fn=process_reels,
    inputs=[
        gr.File(label="Your Instagram export (JSON)"),
        gr.Textbox(label="Gemini API key", type="password"),
        gr.Number(label="Limit to first N reels (0 = no limit)", value=50),
        gr.Checkbox(label="Also download & transcribe video (slower, more accurate)", value=True),
    ],
    outputs=[
        gr.File(label="Download category docs (.zip)"),
        gr.Textbox(label="Summary"),
    ],
    title="Instagram Saved Reels Categorizer",
    description=(
        "Upload your Instagram data export and get a Word doc per category. "
        "Runs on Colab's free compute, so video transcription is enabled by default."
    ),
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2bdea7792617b8f754.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
